# Online Retail II — Exploration

This notebook explores the UCI Online Retail II dataset before any cleaning or analysis. The goal is to understand the structure of the data, identify data quality issues, and produce a cleaning plan for Stage 2.

**Source:** UCI Machine Learning Repository — Online Retail II  
**Coverage:** December 2009 to December 2011  
**File:** `data/raw/online_retail_II.xlsx` (two sheets, one per year)

In [1]:
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 3.0.2


## Loading the data

The file contains two sheets — one per year. I load both and concatenate them into a single DataFrame so the full two-year history is in one place.

In [2]:
# Load both yearly sheets and combine them into one DataFrame
sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

print("Shape:", df.shape)
df.info()

Shape: (1067371, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


## Missing values

Checking the percentage of missing values per column to understand the scale of data quality issues before going deeper.

In [3]:
# Calculate the percentage of missing values per column
(df.isna().sum() / len(df) * 100).round(2)

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64

## Investigating the Invoice column

The `Invoice` column has dtype `object`, which means it can contain any Python type. Before using any string methods, I want to confirm what's actually in there.

In [4]:
# Check the underlying Python types stored in the Invoice column
df["Invoice"].apply(type).value_counts()

Invoice
<class 'int'>    1047871
<class 'str'>      19500
Name: count, dtype: int64

The column contains a mix of `int` and `str` values — numeric-only invoices were stored as integers, while invoices with letter prefixes were stored as strings. This would cause `.str` operations to silently fail on the integer rows. I'll cast the whole column to string to fix this.

In [5]:
# Convert Invoice values to string and confirm the conversion
df["Invoice"] = df["Invoice"].astype(str)
df["Invoice"].apply(type).value_counts()

Invoice
<class 'str'>    1067371
Name: count, dtype: int64

## Invoice prefixes

Now that `Invoice` is uniformly string, I can profile the first character of every invoice to find non-standard prefixes (cancellations, adjustments, etc.).

In [6]:
# Check invoice prefixes to identify cancellations and other non-standard records
df["Invoice"].str[0].value_counts()

Invoice
5    939382
4    108489
C     19494
A         6
Name: count, dtype: int64

Three categories appear:

- Digit prefixes (`4`, `5`) — normal sequential invoice numbers.
- `C` — 19,494 rows, likely cancellations (standard retail convention).
- `A` — 6 rows, unknown. Worth inspecting individually given how few there are.

## Inspecting the A-prefix rows

Only 6 rows, so I can view them all directly.

In [7]:
# Filter A-prefix invoice rows
a_invoices = df[df["Invoice"].str.startswith("A")]
print(a_invoices.shape)
print(a_invoices.to_string())

(6, 8)
        Invoice StockCode      Description  Quantity         InvoiceDate     Price  Customer ID         Country
179403  A506401         B  Adjust bad debt         1 2010-04-29 13:36:00 -53594.36          NaN  United Kingdom
276274  A516228         B  Adjust bad debt         1 2010-07-19 11:24:00 -44031.79          NaN  United Kingdom
403472  A528059         B  Adjust bad debt         1 2010-10-20 12:04:00 -38925.87          NaN  United Kingdom
825443  A563185         B  Adjust bad debt         1 2011-08-12 14:50:00  11062.06          NaN  United Kingdom
825444  A563186         B  Adjust bad debt         1 2011-08-12 14:51:00 -11062.06          NaN  United Kingdom
825445  A563187         B  Adjust bad debt         1 2011-08-12 14:52:00 -11062.06          NaN  United Kingdom


All six `A` rows share the same signature: `StockCode = "B"`, `Description = "Adjust bad debt"`, `Customer ID = NaN`, and prices in the tens of thousands. These are accounting adjustments (bad-debt write-offs), not customer transactions. They'll be removed in cleaning.

## Inspecting the C-prefix rows (cancellations)

19,494 rows — too many to view individually. Looking at a small sample first to confirm these follow the expected cancellation pattern.

In [8]:
# Filter C-prefix invoice rows
cancellations = df[df["Invoice"].str.startswith("C")]
print("Shape of cancellations:", cancellations.shape)
print(cancellations.head(5).to_string())

Shape of cancellations: (19494, 8)
     Invoice StockCode                    Description  Quantity         InvoiceDate  Price  Customer ID    Country
178  C489449     22087       PAPER BUNTING WHITE LACE       -12 2009-12-01 10:33:00   2.95      16321.0  Australia
179  C489449    85206A   CREAM FELT EASTER EGG BASKET        -6 2009-12-01 10:33:00   1.65      16321.0  Australia
180  C489449     21895  POTTING SHED SOW 'N' GROW SET        -4 2009-12-01 10:33:00   4.25      16321.0  Australia
181  C489449     21896             POTTING SHED TWINE        -6 2009-12-01 10:33:00   2.10      16321.0  Australia
182  C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12 2009-12-01 10:33:00   2.95      16321.0  Australia


The sample matches the expected pattern: negative quantities, positive prices, real product codes, and populated Customer IDs. Multiple rows can share one cancellation invoice — a single return can span several line items.

## Cross-validating C-prefix vs. negative quantity

Both signals should identify cancellations. A 2x2 crosstab tells me whether they agree — any off-diagonal count is a data quality finding worth investigating.

In [9]:
# Compare C-prefix invoices with negative quantities to find mismatches
is_cancel_invoice = df["Invoice"].str.startswith("C")
is_negative_qty = df["Quantity"] < 0

pd.crosstab(
    is_cancel_invoice,
    is_negative_qty,
    rownames=["Invoice starts with C"],
    colnames=["Quantity < 0"]
)

Quantity < 0,False,True
Invoice starts with C,,
False,1044420,3457
True,1,19493


Two anomalies to investigate:

- **1 row** is C-prefix but has non-negative quantity.
- **3,457 rows** have negative quantity but no C-prefix.

## Anomaly 1 — The C-prefix row with non-negative quantity

Just one row, so I can view it directly.

In [10]:
# Inspect the C-prefix row with a non-negative quantity
weird_c_row = df[
    df["Invoice"].str.startswith("C") &
    (df["Quantity"] >= 0)
]
print(weird_c_row.to_string())

       Invoice StockCode Description  Quantity         InvoiceDate   Price  Customer ID         Country
76799  C496350         M      Manual         1 2010-02-01 08:24:00  373.57          NaN  United Kingdom


Same admin-row signature as the A invoices: `StockCode = "M"`, `Description = "Manual"`, `Customer ID = NaN`. Another manual accounting entry, not a real transaction.

## Anomaly 2 — Negative-quantity rows without a C-prefix

3,457 rows. I'll look at a sample first, then test a hypothesis about the full set.

In [11]:
# Inspect negative-quantity rows that do not have a C-prefix invoice
neg_qty_non_c = df[
    (~df["Invoice"].str.startswith("C")) &
    (df["Quantity"] < 0)
]
print("Shape:", neg_qty_non_c.shape)
print(neg_qty_non_c.head(5).to_string())

Shape: (3457, 8)
     Invoice StockCode   Description  Quantity         InvoiceDate  Price  Customer ID         Country
263   489464     21733  85123a mixed       -96 2009-12-01 10:52:00    0.0          NaN  United Kingdom
283   489463     71477         short      -240 2009-12-01 10:52:00    0.0          NaN  United Kingdom
284   489467    85123A   21733 mixed      -192 2009-12-01 10:53:00    0.0          NaN  United Kingdom
470   489521     21646           NaN       -50 2009-12-01 11:44:00    0.0          NaN  United Kingdom
3114  489655     20683           NaN       -44 2009-12-01 17:26:00    0.0          NaN  United Kingdom


Every sample row shows `Price = 0.00` and `Customer ID = NaN`, with descriptions that look like warehouse shorthand ("85123a mixed", "short", or blank). These look like internal inventory adjustments — stock corrections, not customer activity. Verifying that pattern holds across all 3,457 rows before concluding.

In [12]:
# Verify the inventory-adjustment hypothesis across all 3,457 rows
print("Non-zero price count:", (neg_qty_non_c["Price"] != 0).sum())
print("Populated Customer ID count:", neg_qty_non_c["Customer ID"].notna().sum())

Non-zero price count: 0
Populated Customer ID count: 0


Both counts are zero, so the pattern is consistent across the whole subset. All 3,457 rows are inventory adjustments with zero price and no customer — internal stock corrections, not customer transactions.

## Stage 1: Exploration — Summary and Cleaning Plan 

### Dataset Overview

The raw dataset contains **1,067,371 rows and 8 columns**, loaded from two yearly sheets: **2009–2010** and **2010–2011**.

**Columns:** `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`

The dataset covers transactions from **December 2009 to December 2011** and comes from the **UCI Machine Learning Repository — Online Retail II** dataset.

### Main Findings

The biggest issue is missing `Customer ID` values. In total, **243,007 rows (22.8%)** do not have a customer attached. Since this project focuses on customer behaviour, these rows cannot be used for RFM, cohort, or churn analysis.

A smaller issue is missing `Description` values. This affects **4,382 rows (0.4%)**, so I will leave this for now unless it causes problems later.

There are also a few data type issues. `Customer ID` is stored as `float64` because pandas cannot store missing values in a normal integer column, so it defaults to float. The `Invoice` column had mixed types: numeric invoices were read as numbers, while invoices with prefixes were read as text. I converted `Invoice` to string so prefix checks work reliably.

The invoice prefixes revealed several non-standard records:

- **6 `A`-prefix rows**: bad-debt accounting adjustments, not customer transactions.
- **1 `C`-prefix row with positive quantity**: a manual adjustment, not a normal cancellation.
- **3,457 negative-quantity rows without a `C` prefix**: likely inventory adjustments because they have `Price = 0.00` and missing `Customer ID`.
- **19,493 standard `C`-prefix rows**: cancellation or refund records that need separate handling.

### Why This Matters

The non-transactional rows identified so far all have missing `Customer ID` values. This includes the bad-debt adjustments, the manual adjustment row, and the inventory-adjustment rows.

Because this project focuses on customer behaviour, rows without `Customer ID` cannot be linked to a customer. Dropping rows with missing `Customer ID` should therefore remove unusable customer records and also exclude the non-transactional adjustment rows found during exploration.

### Cleaning Plan for Stage 2

1. **Convert `Invoice` to string type**  
   This keeps prefix checks reliable.

2. **Drop rows with missing `Customer ID`**  
   These rows cannot be used for customer-level analysis. From what I saw during exploration, this step should also remove the adjustment rows that are not real customer transactions.

3. **Handle `C`-prefix cancellations**  
   These are real cancellation records, so I should not remove them without checking their matching purchases. The likely approach is to remove both the cancellation row and the matching original purchase, so revenue reflects completed sales only.

4. **Check for any remaining invalid quantities or prices**  
   After handling cancellations, I will check for any remaining rows with `Quantity <= 0` or `Price <= 0`.

5. **Convert `Customer ID` to a cleaner type**  
   Once missing values are removed, I can convert `Customer ID` from `float64` to a more suitable integer or string format.

6. **Create a `Revenue` column**  
   Revenue will be calculated as `Quantity * Price` for RFM monetary value and later analysis.

7. **Save the cleaned dataset**  
   Save the cleaned file in `data/processed/` for the next stage of the project.

### Expected Cleaned Dataset

After cleaning, I expect the dataset to contain approximately:

- **800,000 rows**
- **5,900 unique customers**
- **37 countries**
- Transactions from **December 2009 to December 2011**

The row estimate is mainly based on removing the **243,007 rows without `Customer ID`**, then handling cancellations and any remaining invalid rows. I will confirm the final numbers after Stage 2.